# Introduction
In the world of hospitality, customer reviews are a goldmine of insights — but with thousands of reviews scattered across text, voice, video, and images, it's hard for businesses to make sense of them all.

An intelligent pipeline that takes in text reviews and outputs a structured, grounded summary — powered by RAG, embeddings, one shot prompting and text understanding.

This project builds a Smart Review Summarizer that:

# Phase 1 (By April 20th ) - Gen AI capabilities incorporated
1. Structured output/JSON model: Current program is storing the JSON structure in a output file.
2. Document understanding : The program currently reads the reviews from the sample data created in the the excel file.
3. Embeddings : Sentence transformers are used in this project - all-miniLM-L6-v2
4. Retrieval augmented genration (RAG) : Using FLAN-T5 model for genrating relevant summaries by retriving passges from the vector store based on
   the query
5. Vector store : FAISS index6. 
6. Classification of reviews based on
        cleanliness, location, value, service,food_availability,
        crowd_level, pricing,queue_fairness ,park_experience,
        emotional_impact , attractions , overall_experience

# Phase 2 (After Apr 20th)
1. Using agents to interact with live reviews using Google API
2. Using Context Cache
3. MLOps : Making the project production mode


# GitHub

https://github.com/prakashpillai/LearningGenAI.git




# Use Case
Imagine a hotel manager trying to understand guest feedback from Booking.com, Google reviews, and Trip advisor messages. Instead of reading hundreds of reviews manually, our system:

Summarizes guest experiences (cleanliness, location, service, value, etc.)
Categorizes pros/cons from real reviews
Grounds summaries in similar past reviews using RAG





# Step 1 : Install required libraries

In [1]:
# Install the required libraries for embedding text,Hugging face core library
!pip install -q sentence-transformers transformers faiss-cpu scikit-learn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 36.2 MB/s eta 0:00:00:00:0100:01


# Step 2 : Importing the required libraries

In [2]:
import json
import re
import csv
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from transformers import T5Tokenizer, T5ForConditionalGeneration
# for fast vector similarity search
import faiss

In [3]:
from transformers import pipeline

zero_shot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

candidate_labels = [
    "cleanliness", "location", "value", "service", "food_availability",
    "crowd_level", "pricing", "queue_fairness", "park_experience",
    "emotional_impact", "attractions", "overall_experience"
]

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cpu


### Step 3 : Load a sample Dataset from Excel
Creating a simple dataset of reviews using Excel and uploaded the file in the dataset.Reading the excel using Pandas.

In [4]:
# Load the Excel file containing user reviews
# Remove any rows with missing review text to ensure clean input
# Extract reviews as a list
#loading sample data from file in the pandas dataframe.
df = pd.read_excel('/kaggle/input/test-data/Dataset_Kaggle_project.xlsx', engine='openpyxl')
#Removes rows with missing values
df = df.dropna(subset=["Review"])
texts = df['Review'].tolist()
df.head()


,Review,Scoring
0,Absolutely fantastic,5
1,"Absolutely magical, our grandchildren were tot...",5
2,After spending hours standing in the long over...,1
3,Best place on earth,5
4,Cool,5


# Step 4: Load the pre-trained model

In [5]:
# Load SentenceTransformer model for embeddings
# Load FLAN-T5 for prompt-based text generation (cluster labeling)
model_name = 'google/flan-t5-large'
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generate text embeddings for each review using pre-trained model and store them in a FAISS vector database.

# Step 5 # Create the embeddings and index for similarity search (FAISS index)

In [6]:
embeddings = embed_model.encode(texts, show_progress_bar=True)
dimension = embeddings.shape[1]
# Build a FAISS index for fast similarity querying
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

# Step 6 : Clause level categorization - Text split and preprocessing

In [8]:
# Utility to break down long reviews into shorter, semantically meaningful clauses
# Helps improve clustering and labeling accuracy
def split_review_into_clauses(review):
    raw_clauses = re.split(r'[.!?;]+', review)
    return [c.strip() for c in raw_clauses if len(c.strip().split()) >= 5]

# Step 7: Clause Categorization and sentiment detection

In [ ]:
# Load external rule files (uploaded to Kaggle)
with open("/kaggle/input/d/taniaparisi/sentiment-overrides/category_rules.json") as f:
    CATEGORY_RULES = json.load(f)

# Step 4# RAG setup (Retrieval Augmented generation) 

RAG Combines 
1. Retrieval (Search from vector store) 
2. Generation(use the language model to answer based on retrieved data)

In [8]:
#previously we used the embed model to convert the reviews into numeric form
#then convert the query into vector form
#find the closest match using FAISS . Now going to use the Generative model(FLAN-T5) 
#which will assist to perform the text summarization
#based on the simple query

#Import FLAN-T5 model from Hugging Face.
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "google/flan-t5-small"
#converts the text into tokens that model understands.
tokenizer = T5Tokenizer.from_pretrained(model_name)
#T%ConditionalGenration is the actual FLAN-T5 model, used for tasks like summarization or Q & A.
model = T5ForConditionalGeneration.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

# Step 5# Few shot prompt Design

In [ ]:
#just putting some string.
query ='Disneyland Parks'
# When you generate embbedings using "SentenceTransformer" they return as NumpY array by default. 
# Converting our string to a query vector. embed_model has all the reviews which we shared through dataset.
query_vector = embed_model.encode([query], convert_to_numpy=True).reshape(1, -1)

#previously we used the embed model to convert the review into numeric form
#convert the query into vector form
#find the closest match using FAISS . Now going to use the Generative model(FLAN-T5) which will assist to give answers
#based on the simple query

#Search for top 3 similar reviews
k = 3
distances , indices = index.search(query_vector,k)
# The row numbers of the most similar reviews
print(indices)
#how close those reviews are to the query
print(distances)
# This will take values of 0,2,3 indices from DF and store in top_k_reviews as a list.
top_k_reviews =  [df.iloc[idx]['Review'] for idx in indices[0]]
print(top_k_reviews)

# === Define full RAG to JSON function ===

def rag_to_json(query, k):
    # Retrieve top documents
    query_vector = embed_model.encode([query], convert_to_numpy=True).reshape(1, -1)
    distances, indices = index.search(query_vector, k)
    top_k_reviews = [df.iloc[idx]['Review'] for idx in indices[0]] 

# Prepare context from reviews
    context = " ".join([review[:300] for review in top_k_reviews])
    
# Generate answer using FLAN-T5
    prompt = f"Summarize the following  hotel reviews : {context}"
    inputs = tokenizer(prompt,return_tensors='pt',truncation=True, max_length=512)
    outputs = model.generate(**inputs,max_new_tokens=100)

    summary = tokenizer.decode(outputs[0],skip_special_tokens=True)
     # Generate embeddings for the top 3 reviews
    k_review_embeddings = embed_model.encode(top_k_reviews, show_progress_bar=True, convert_to_numpy=True)

    # Convert embeddings (numpy array) to a list for JSON serialization
    embeddings_list = k_review_embeddings.tolist()
    
    
    print(f"Answer :{summary}")
      # Format as JSON
    result = {
        "query": query,
        "summary": summary,
        "top_k_reviews": top_k_reviews,
        "embeddings" : embeddings_list
        }    
    #print(json.dumps(result, indent=2, ensure_ascii=False))    
    return result
    
output=rag_to_json("Disneyland Parks",3)


# Step 6# JSON Summary Generation

Exporting the JSON to a external file name ''

In [22]:
# Save the output from the prev step and save as a JSON file
with open("summary_output.json", "w", encoding="utf-8") as f: json.dump(output, f, indent=2, ensure_ascii=False)

print("### Summary exported to 'summary_output.json' successfully ####")

### Summary exported to 'summary_output.json' successfully ####


# Adding visualization to the review data based on JSON genrated.



# Phase 2 - Future product enchancments
1. To review the images.
2. Use Agents to interact with live reviews using API
3. Adding the MLOps to make the product